# csp_workflow_mp — Visualise benchmark predictions

Reads `results/benchmark_raw.csv` (500 LOO samples × 3 strategies = 1500 rows) and plots:

1. Success rate per stage per strategy (substitution → relax → SG match → StructureMatcher match).
2. RMSD distribution per strategy for converged predictions.
3. Per-`n_elements` success bars to show where the pipeline struggles.

Strategies:
- **unconstrained** — descriptor-only retrieval, no symmetry filter.
- **sg_only** — descriptor + space-group filter (the canonical mode).
- **sg_ps** — descriptor + (SG, Pearson-prefix) filter (more restrictive).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

results_dir = Path(__file__).resolve().parent.parent / 'results' \
    if '__file__' in globals() else Path('../results')
df = pd.read_csv(results_dir / 'benchmark_raw.csv')
print(f'{len(df)} rows  ·  strategies: {sorted(df.strategy.unique())}')
df.head()

## 1. Per-stage success rate

In [ ]:
stages = ['sub_success', 'relax_converged', 'sg_match', 'sm_match']
rates  = (df.groupby('strategy')[stages].mean() * 100).round(1)
rates

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
rates.T.plot(kind='bar', ax=ax, width=0.78)
ax.set_ylabel('Success rate (%)')
ax.set_xlabel('Pipeline stage')
ax.set_title('Cumulative success at each stage, per strategy')
ax.set_xticklabels(['Substitution', 'Relax', 'SG match', 'SM match'], rotation=0)
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
ax.legend(title='Strategy', frameon=False)
plt.tight_layout()
plt.show()

## 2. RMSD distribution (relax-converged subset)

RMSD is `pymatgen.analysis.structure_matcher.StructureMatcher.get_rms_dist`, computed only for samples that passed both substitution and relaxation.

In [ ]:
rmsd_df = df.dropna(subset=['rmsd_angstrom'])
rmsd_df.groupby('strategy')['rmsd_angstrom'].describe().round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.linspace(0, max(0.5, rmsd_df.rmsd_angstrom.quantile(0.95)), 30)
for strategy, sub in rmsd_df.groupby('strategy'):
    ax.hist(sub.rmsd_angstrom, bins=bins, alpha=0.45, label=strategy)
ax.set_xlabel('RMSD (Angstrom)')
ax.set_ylabel('Count')
ax.set_title('RMSD vs ground-truth, per strategy (relax-converged)')
ax.legend(title='Strategy', frameon=False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Success rate by number of elements

Higher-arity formulas are harder: the template pool gets sparser and the substitution engine has to enumerate more role-assignments.

In [ ]:
by_n = (df.groupby(['n_elements','strategy'])['sm_match']
          .mean().mul(100).round(1)
          .unstack('strategy'))
by_n

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
by_n.plot(kind='bar', ax=ax, width=0.78)
ax.set_xlabel('Number of elements in target formula')
ax.set_ylabel('StructureMatcher match (%)')
ax.set_title('SM match vs formula complexity, per strategy')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)
ax.legend(title='Strategy', frameon=False)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Volume change before / after relaxation

`vol_change` = |V_relaxed - V_template| / V_template. A spike above ~0.30 usually means the substitution chose a template too far away in chemistry; these are flagged via `vol_filtered`.

In [ ]:
vol_df = df.dropna(subset=['vol_change'])
fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.linspace(0, vol_df.vol_change.quantile(0.99), 30)
for strategy, sub in vol_df.groupby('strategy'):
    ax.hist(sub.vol_change, bins=bins, alpha=0.45, label=strategy)
ax.axvline(0.30, color='k', ls='--', lw=0.8, label='vol_filtered threshold')
ax.set_xlabel('|V_pred - V_template| / V_template')
ax.set_ylabel('Count')
ax.set_title('Volume change distribution, per strategy')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()